<a href="https://colab.research.google.com/github/royalsflush/hackernews_vs_googletrends/blob/main/Hacker_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.cloud import bigquery
import pandas as pd

In [3]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

project_id = 'data-mining-project-505611'
client = bigquery.Client(project=project_id)

Authenticated


In [ ]:
# Create story view
query = """
CREATE OR REPLACE VIEW hackernews_royalsflush.hackernews_story AS (
  SELECT title,
        url,
        text,
        `by`,
        score,
        `timestamp`,
        id,
        descendants
  FROM `bigquery-public-data.hacker_news.full`
  WHERE type = 'story'
  AND `by` IS NOT NULL
  AND dead IS NULL
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=4d24bc03-7e50-46e5-9253-d9b477ce5ad7>


In [ ]:
# Create comments view
query = """
CREATE OR REPLACE VIEW hackernews_royalsflush.hackernews_comment AS (
  SELECT text,
       `by`,
       `timestamp`,
       id,
       parent,
  FROM `bigquery-public-data.hacker_news.full`
  WHERE type = 'comment'
  AND `by` IS NOT NULL
  AND dead IS NULL
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=79767495-4695-4150-9ecf-a7e9c7e79df2>


In [ ]:
# Create story <-> comment ID mapping
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_comment_story_mapping AS (
  WITH RECURSIVE
  comment_id_pairs AS (
    SELECT id, parent
    FROM `data-mining-project-505611.hackernews_royalsflush.hackernews_comment`
  ),
  story_ids AS (
    SELECT id
    FROM `data-mining-project-505611.hackernews_royalsflush.hackernews_story`
  ),
  R AS (
    (SELECT id, parent AS ancestor from comment_id_pairs)
    UNION ALL (
      SELECT R.id, comment_id_pairs.parent AS ancestor
      FROM R
      INNER JOIN comment_id_pairs ON ancestor = comment_id_pairs.id
    )
  )
  SELECT R.id, ancestor
  FROM R
  INNER JOIN story_ids
  ON story_ids.id = R.ancestor
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=a27ad46c-e5bf-4559-a166-49e495f1b3d9>


In [ ]:
!pip install requests

In [6]:
import concurrent.futures
import requests
import threading
import time
from tqdm import tqdm

query = """
SELECT id, url
FROM `hackernews_royalsflush.hackernews_story`
WHERE url IS NOT NULL
LIMIT 10000;
"""

df = client.query(query).to_dataframe()
df.set_index('id', inplace=True)
df_lock = threading.Lock()
print(df)

headers = {
    'User-Agent': 'My User Agent 1.0',
    'From': 'youremail@domain.example'  # This is another valid field
}

# Modified example from ThreadPoolExecutor docs
def load_url(url, timeout):
    return requests.get(url, timeout=timeout, headers=headers)

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=100) as executor:
    # Start the load operations and mark each future with its URL
    future_to_url_id = {executor.submit(load_url, row['url'], 60): id for id, row in df.iterrows()}
    for future in tqdm(concurrent.futures.as_completed(future_to_url_id), total=len(df)):
        id  = future_to_url_id[future]

        try:
            data = future.result()
        except Exception as exc:
            with df_lock:
                df.loc[id, 'content'] = ""
                df.loc[id, 'url_error'] = True
        else:
            with df_lock:
                df.loc[id, 'content'] = data.text
                df.loc[id, 'url_error'] = False
print(f"URL fetching time: {time.time() - start}")

start = time.time()
load_job = client.load_table_from_dataframe(df, 'hackernews_royalsflush.hackernews_story_content')
print(f"BigQuery load time: {time.time() - start}")
print(resp)

                                                       url
id                                                        
8846615  https://www.cryptocoinsnews.com/bitstamp-resum...
8846628  https://www.cryptocoinsnews.com/bitstamp-resum...
8846665  http://www.theregister.co.uk/2014/12/23/google...
8846673                                 http://pgpasc.org/
8846674  http://data.khanacademy.org/2015/01/i-need-ans...
...                                                    ...
8833924        http://en.wikipedia.org/wiki/Murphy%27s_law
8833926  https://play.google.com/store/apps/details?id=...
8833942      http://www.sciencemag.org/content/347/6217/75
883397   http://www.sciencedaily.com/releases/2009/10/0...
8834005                   https://github.com/racker/falcon

[10000 rows x 1 columns]


100%|██████████| 10000/10000 [16:08<00:00, 10.32it/s]


URL fetching time: 980.6365852355957
BigQuery load time: 61.992111682891846
LoadJob<project=data-mining-project-505611, location=US, id=353ff8db-b962-4aa3-ae95-d8f49077919f>


In [8]:
# Aggregates all the information needed to calculate the popularity score
# and the topics
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_payload AS (
  SELECT story.id AS story_id,
        CONCAT("https://news.ycombinator.com/item?id=", id) AS story_link,
        score AS story_score,
        story.timestamp AS story_timestamp,
        CONCAT(title, "\\n",
                IF(text IS NOT NULL, text, ""), "\\n",
                IFNULL(ARRAY_TO_STRING(comment.comment_text, "\\n", ""), "")) AS payload,
        comment.comment_ids,
        comment.comment_timestamps
  FROM `hackernews_royalsflush.hackernews_story` AS story
  LEFT JOIN (
    SELECT ARRAY_AGG(comment.id) AS comment_ids,
          ARRAY_AGG(text) AS comment_text,
          ARRAY_AGG(`timestamp`) AS comment_timestamps,
          mapping.ancestor AS story_id
    FROM `hackernews_royalsflush.hackernews_comment` AS comment
    JOIN `hackernews_royalsflush.hackernews_comment_story_mapping` AS mapping
    ON mapping.id = comment.id
    GROUP BY mapping.ancestor
  ) AS comment
  ON comment.story_id = story.id
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=f2d51cfd-039e-426e-b085-8f8942153f8b>


In [15]:
# Calculates the popularity for a particular story bucketed by month.
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_popularity AS (
  SELECT story_id,
        month,
        SUM(popularity) AS popularity
  FROM (
    SELECT story_id,
          DATETIME_TRUNC(story_timestamp, MONTH) AS month,
          story_score AS popularity,
          "story" AS type
    FROM `hackernews_royalsflush.hackernews_payload` AS story_payload
    UNION ALL
    SELECT story_id,
          DATETIME_TRUNC(comment_timestamp, MONTH) as month,
          1 AS popularity,
          "comment" AS type
    FROM `hackernews_royalsflush.hackernews_payload` AS story_payload,
      UNNEST(comment_timestamps) AS comment_timestamp
  )
  GROUP BY story_id, month
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=cd987144-cf87-4ec4-99c3-e495190ca0d5>


In [10]:
!pip install pytrends-modern

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 761.1 kB/s eta 0:00:00


In [14]:
from pytrends_modern import TrendReq

pytrends = TrendReq(hl='en-US', tz=0)
pytrends.build_payload(
    kw_list=['Python', 'JavaScript'],
    timeframe='all',
    geo='', # Default is world
)

interest_df = pytrends.interest_over_time()
related = pytrends.related_queries()

print(interest_df)
print(related)

            Python  JavaScript  isPartial
date                                     
2004-01-01      24          93      False
2004-02-01      23          94      False
2004-03-01      23         100      False
2004-04-01      24          98      False
2004-05-01      22          87      False
...            ...         ...        ...
2026-04-01      75          14      False
2026-05-01      84          17      False
2026-06-01      88          19      False
2026-07-01      65          14      False
2026-08-01      48           8       True

[272 rows x 3 columns]
{'Python': {'top':                  query  value
0           python for    100
1          python list     75
2          python code     43
3       install python     34
4            python if     34
5         monty python     30
6        online python     30
7       what is python     29
8      download python     28
9       python windows     24
10              pandas     23
11        python array     23
12       pandas pytho